# Chapter 11 — Persistent Time Series: Unit Roots & Cointegration
## University of Neuchâtel — Python for Econometrics

This chapter covers:
1. Spurious regressions with nonstationary data
2. Unit root testing (ADF test)
3. First differencing to achieve stationarity
4. Cointegration and Error Correction Models (ECM)

**Format:** One cell per step. Manual calculation first, then statsmodels, then `np.isclose()` comparison.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.graphics.tsaplots import plot_acf
import matplotlib.pyplot as plt
import wooldridge as woo
import os

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

## 1 — Load Data: `intdef`

The `intdef` dataset contains annual US data on:
- `i3`: 3-month T-bill interest rate
- `inf`: CPI inflation rate
- `def`: federal budget deficit (% of GDP)
- `year`: 1948–2003

Both `i3` and `inf` appear to be **persistent (nonstationary)** — trending together.

In [ ]:
df = woo.dataWoo('intdef')
print('Shape:', df.shape)
print(df[['year','i3','inf','def']].head(10))

In [ ]:
print('Descriptive statistics:')
print(df[['i3','inf']].describe().round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(df['year'], df['i3'], label='i3 (T-bill rate)', linewidth=2)
ax.plot(df['year'], df['inf'], label='inf (inflation)', linewidth=2, linestyle='--')
ax.set_xlabel('Year')
ax.set_ylabel('Percent')
ax.set_title('US Interest Rate and Inflation, 1948-2003')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch11_01_i3_inf.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2 — Spurious Regression

**What is a spurious regression?** When two series both have **unit roots** (are I(1)),
regressing one on the other often yields high $R^2$ and significant $t$-statistics —
but this is **purely coincidental**. There is no real causal relationship.

**Model:** $i3_t = \beta_0 + \beta_1 \cdot inf_t + u_t$

Even if $i3$ and $inf$ are unrelated random walks, we may find significant results.

### Step 2.1 — Manual OLS: i3 ~ inf

In [ ]:
n = len(df)
x = df['inf'].values
y = df['i3'].values
x_dev = x - x.mean()
y_dev = y - y.mean()
b1_manual = np.sum(x_dev * y_dev) / np.sum(x_dev**2)
b0_manual = y.mean() - b1_manual * x.mean()
print('Manual OLS: i3 ~ inf')
print(f'  b0 = {b0_manual:.4f}')
print(f'  b1 = {b1_manual:.4f}')

### Step 2.2 — Statsmodels OLS

In [ ]:
res_spur = smf.ols('i3 ~ inf', data=df).fit()
print(res_spur.summary())

### Step 2.3 — Compare manual vs statsmodels

In [ ]:
match_b0 = np.isclose(b0_manual, res_spur.params['Intercept'])
match_b1 = np.isclose(b1_manual, res_spur.params['inf'])
print(f'b0: manual={b0_manual:.4f}, sm={res_spur.params["Intercept"]:.4f}, match={match_b0}')
print(f'b1: manual={b1_manual:.4f}, sm={res_spur.params["inf"]:.4f}, match={match_b1}')
print(f'R2={res_spur.rsquared:.3f}, p(inf)={res_spur.pvalues["inf"]:.4f}')
print('WARNING: High R2 may be spurious if both series are I(1)!')

## 3 — Unit Root Testing (ADF Test)

**Augmented Dickey-Fuller (ADF) Test:**

$H_0$: series has a unit root (nonstationary, I(1))

$H_1$: series is stationary (I(0))

**Model estimated:** $\Delta y_t = \alpha + \rho y_{t-1} + \sum_{j=1}^p \gamma_j \Delta y_{t-j} + e_t$

- Test stat = $t$-statistic on $\rho$
- Critical values are non-standard (Dickey-Fuller, not Student-$t$)
- If test stat < critical value (very negative), reject $H_0$ → stationary

### Step 3.1 — Simulate: Random Walk vs Stationary AR(1)

In [ ]:
np.random.seed(42)
T = 100
eps = np.random.normal(0, 1, T)
# Random walk: y_t = y_{t-1} + eps_t (unit root, rho=1)
rw = np.cumsum(eps)
# Stationary AR(1): y_t = 0.5*y_{t-1} + eps_t (|rho|<1)
ar1 = np.zeros(T)
for t in range(1, T):
    ar1[t] = 0.5*ar1[t-1] + eps[t]
print('Random walk: mean=', round(rw.mean(),3), ' std=', round(rw.std(),3))
print('AR(1):       mean=', round(ar1.mean(),3), ' std=', round(ar1.std(),3))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
ax1.plot(rw, linewidth=1.5)
ax1.set_title('Random Walk (Unit Root, I(1))')
ax1.set_xlabel('Time')
ax2.plot(ar1, linewidth=1.5, color='orange')
ax2.set_title('Stationary AR(1) (|rho|<1)')
ax2.set_xlabel('Time')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch11_02_rw_vs_ar1.png'), dpi=150, bbox_inches='tight')
plt.show()

### Step 3.2 — ADF test on i3

In [ ]:
adf_i3 = adfuller(df['i3'].dropna(), maxlag=1, autolag=None)
print('ADF Test on i3 (interest rate):')
print(f'  Test statistic: {adf_i3[0]:.4f}')
print(f'  p-value:        {adf_i3[1]:.4f}')
print('  Critical values:')
for key, val in adf_i3[4].items():
    print(f'    {key}: {val:.4f}')
if adf_i3[1] > 0.05:
    print('  CONCLUSION: Fail to reject H0 → i3 likely has a unit root (I(1))')
else:
    print('  CONCLUSION: Reject H0 → i3 is stationary')

### Step 3.3 — ADF test on inf

In [ ]:
adf_inf = adfuller(df['inf'].dropna(), maxlag=1, autolag=None)
print('ADF Test on inf (inflation):')
print(f'  Test statistic: {adf_inf[0]:.4f}')
print(f'  p-value:        {adf_inf[1]:.4f}')
print('  Critical values:', {k: round(v,3) for k,v in adf_inf[4].items()})
if adf_inf[1] > 0.05:
    print('  CONCLUSION: Fail to reject H0 → inf likely has a unit root (I(1))')
else:
    print('  CONCLUSION: Reject H0 → inf is stationary')

## 4 — First Differencing

If a series is I(1), taking the **first difference** makes it I(0) (stationary):

$$\Delta y_t = y_t - y_{t-1}$$

**Why difference?** Regressions on differenced data avoid spurious results.

### Step 4.1 — Compute first differences manually

In [ ]:
df['d_i3']  = df['i3'].diff()
df['d_inf'] = df['inf'].diff()
print('Dataset after adding first differences (first 5 rows):')
print(df[['year','i3','d_i3','inf','d_inf']].head(5))
print('Note: first row is NaN (no previous value)')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12,8))
axes[0,0].plot(df['year'], df['i3'])
axes[0,0].set_title('i3 (levels)')
axes[0,1].plot(df['year'], df['d_i3'])
axes[0,1].set_title('Δi3 (first difference)')
axes[0,1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1,0].plot(df['year'], df['inf'])
axes[1,0].set_title('inf (levels)')
axes[1,1].plot(df['year'], df['d_inf'])
axes[1,1].set_title('Δinf (first difference)')
axes[1,1].axhline(0, color='red', linestyle='--', alpha=0.5)
for ax in axes.flat:
    ax.set_xlabel('Year')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch11_03_levels_vs_diff.png'), dpi=150, bbox_inches='tight')
plt.show()

### Step 4.2 — ADF test on first differences

In [ ]:
d_i3_clean = df['d_i3'].dropna()
adf_di3 = adfuller(d_i3_clean, maxlag=1, autolag=None)
print('ADF Test on Δi3 (first difference):')
print(f'  Test statistic: {adf_di3[0]:.4f}')
print(f'  p-value:        {adf_di3[1]:.4f}')
if adf_di3[1] < 0.05:
    print('  CONCLUSION: Reject H0 → Δi3 is stationary (I(0)) ✓')

### Step 4.3 — Regression in first differences

In [ ]:
df_diff = df[['d_i3','d_inf']].dropna().copy()
print('Data for differences regression (no NaN):')
print(df_diff.head())
print('Shape:', df_diff.shape)

In [ ]:
# Manual OLS: d_i3 ~ d_inf
xd = df_diff['d_inf'].values
yd = df_diff['d_i3'].values
b1_diff = np.sum((xd - xd.mean())*(yd - yd.mean())) / np.sum((xd - xd.mean())**2)
b0_diff = yd.mean() - b1_diff * xd.mean()
print('Manual OLS (first differences): d_i3 ~ d_inf')
print(f'  b0 = {b0_diff:.4f}')
print(f'  b1 = {b1_diff:.4f}')

In [ ]:
res_diff = smf.ols('d_i3 ~ d_inf', data=df_diff).fit()
print(res_diff.summary())

In [ ]:
match_b1d = np.isclose(b1_diff, res_diff.params['d_inf'])
print(f'b1 match: manual={b1_diff:.4f}, sm={res_diff.params["d_inf"]:.4f}, match={match_b1d}')
print('Compare levels vs differences regression:')
print(f'  Levels R2:     {res_spur.rsquared:.3f}')
print(f'  Differences R2:{res_diff.rsquared:.3f}')

## 5 — Cointegration

**Two I(1) series are cointegrated** if a linear combination of them is I(0):

$$y_t - \beta x_t \sim I(0)$$

If cointegrated, it's valid to run regressions in levels (not spurious).
The long-run relationship is real.

**Engle-Granger test:** Run OLS, test if the residuals are I(0) with ADF.

### Step 5.1 — Engle-Granger cointegration test manually

In [ ]:
# Step 1: Regress i3 on inf (cointegrating regression)
res_coint = smf.ols('i3 ~ inf', data=df).fit()
coint_resid = res_coint.resid.values
print('Step 1: cointegrating regression i3 ~ inf')
print(f'  beta = {res_coint.params["inf"]:.4f}')

In [ ]:
# Step 2: ADF test on residuals
adf_resid = adfuller(coint_resid, maxlag=1, autolag=None, regression='nc')
print('Step 2: ADF test on cointegrating residuals:')
print(f'  Test statistic: {adf_resid[0]:.4f}')
print(f'  p-value:        {adf_resid[1]:.4f}')
if adf_resid[1] < 0.1:
    print('  CONCLUSION: Residuals are I(0) → i3 and inf are COINTEGRATED')
else:
    print('  CONCLUSION: Residuals are I(1) → no cointegration (spurious regression)')

### Step 5.2 — Statsmodels cointegration test

In [ ]:
coint_stat, coint_pval, coint_cv = coint(df['i3'], df['inf'])
print('Statsmodels coint() test:')
print(f'  Test statistic: {coint_stat:.4f}')
print(f'  p-value:        {coint_pval:.4f}')
print(f'  Critical values (1%, 5%, 10%): {np.round(coint_cv,4)}')

### Step 5.3 — Error Correction Model (ECM)

In [ ]:
# ECM: if cointegrated, short-run dynamics + long-run equilibrium
# ecm_t-1 = i3_{t-1} - b0 - b1*inf_{t-1}  (deviation from long-run)
df_ecm = df[['d_i3','d_inf','i3','inf']].copy()
df_ecm['ecm_lag'] = (df['i3'] - res_coint.params['Intercept'] - res_coint.params['inf']*df['inf']).shift(1)
df_ecm = df_ecm.dropna()
print('ECM dataset (first 5 rows):')
print(df_ecm[['d_i3','d_inf','ecm_lag']].head())

In [ ]:
res_ecm = smf.ols('d_i3 ~ d_inf + ecm_lag', data=df_ecm).fit()
print(res_ecm.summary())
print('ecm_lag coefficient (error correction speed):', round(res_ecm.params['ecm_lag'],4))
print('Negative ecm_lag → corrects back to long-run equilibrium')

## 6 — Summary

| Test/Model | Series | Result |
|---|---|---|
| ADF (levels) | i3, inf | Likely I(1) — unit root |
| ADF (diff) | Δi3, Δinf | I(0) — stationary |
| Levels regression | i3 ~ inf | High R² — possibly spurious |
| Differences regression | Δi3 ~ Δinf | Lower R² — valid |
| Cointegration | i3 & inf | Test whether long-run relationship exists |

In [ ]:
summary = pd.DataFrame({
    'Model': ['Spurious (levels)', 'First differences', 'ECM'],
    'b1': [res_spur.params['inf'], res_diff.params['d_inf'], res_ecm.params['d_inf']],
    'R2': [res_spur.rsquared, res_diff.rsquared, res_ecm.rsquared]
})
print(summary.round(4).to_string(index=False))
summary.to_csv(os.path.join(output_dir,'table_ch11_summary.csv'), index=False)
print('Saved table_ch11_summary.csv')